# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring a FAIR^2 dataset—**Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells**—using the `mlcroissant` library. We'll walk through loading the Croissant schema, exploring available record sets, examining field definitions via their `@id`s, and running initial analyses and visualizations.

### Dataset Source

The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and available records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD Schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Instantiate a Croissant Dataset (loads and parses metadata only; does not download data yet)
dataset = mlc.Dataset(url)

# Print Dataset metadata
metadata = dataset.metadata
print("Title:", getattr(metadata, 'name', None))
print("Description:", getattr(metadata, 'description', None))
print("Version:", getattr(metadata, 'version', None))
print("License:", getattr(metadata, 'license', None))

## 2. Data Overview

Review available record sets and fields with their `@id` values.

We'll inspect the record sets, fields (columns), and their unique Croissant `@id`s so you can reference these for further data access and manipulations.

In [ ]:
# View available record sets and their @ids
record_sets = dataset.record_sets
print("Available record sets:")
for rs in record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}, Description: {rs.description}")

# For this dataset, let's inspect the fields/columns for each record set
for rs in record_sets:
    print(f"\nFields for RecordSet '{rs.name}' (@id={rs.id}):")
    for field in rs.fields:
        print(f"  - {field.name} (@id={field.id}, dataType={field.data_type})")

## 3. Data Extraction

Let's extract and load the main record set(s) into a DataFrame for analysis. We will reference record sets, fields, and columns by their `@id`.

First, let’s collect all record set `@id`s, and then load each as a pandas DataFrame using `mlcroissant`.

In [ ]:
# Store a mapping of RecordSet @ids to names for convenience
record_set_ids = [rs.id for rs in dataset.record_sets]
print("RecordSet @ids found:", record_set_ids)

dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading data for RecordSet {rs_id} ...")
    # Get all records for this record set
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  -> Columns in DataFrame:", df.columns.tolist())
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)

Let’s select a **numeric field** for analysis. We will reference this field by its column `@id` (as seen in cell above), and perform basic summary and transformation operations such as filtering, normalization, and grouping.

In [ ]:
# Example: Select a record set (use the first available if unsure)
main_record_set_id = record_set_ids[0]
main_df = dataframes[main_record_set_id]
print(f"Columns of record set '{main_record_set_id}':", main_df.columns.tolist())

# Pick a numeric field/column by its name (for this dataset, e.g., 'MW' or 'Coverage (%)')
# Replace with the appropriate field @id for your dataset; for demo, use first numeric column
numeric_field = None
for col in main_df.columns:
    if pd.api.types.is_numeric_dtype(main_df[col]):
        numeric_field = col
        print(f"Using numeric field: {numeric_field}")
        break
if numeric_field is None:
    raise ValueError("No numeric fields found for analysis!")

# Apply a value threshold for filtering
threshold = main_df[numeric_field].mean() if main_df[numeric_field].dtype != object else 0
filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records where '{{}}' > {{:.2f}}: {{}}".format(numeric_field, threshold, len(filtered_df)))

# Normalize the numeric field (Z-score)
filtered_df[numeric_field + "_zscore"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, numeric_field + "_zscore"]].head())

# Group by a field if one exists (for example: 'accession' or a categorical field)
group_field = None
for col in main_df.columns:
    if pd.api.types.is_object_dtype(main_df[col]) and col != numeric_field:
        group_field = col
        print(f"Grouping by field: {group_field}")
        break
if group_field:
    grouped = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(grouped.head())

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the selected numeric field and (if grouping is available) show group-level means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

# Histogram of the numeric field for the filtered dataframe
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[numeric_field], kde=True)
plt.title(f"Distribution of '{numeric_field}' (filtered > mean)")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If group_field was found, boxplot by group
if group_field and len(filtered_df[group_field].unique()) <= 20:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion

In this notebook, we loaded a FAIR^2 dataset (referencing all entities by their `@id`s) using the `mlcroissant` library, listed available record sets and fields, extracted data as DataFrames, and performed initial exploration and visualization on one numeric field. With Croissant's structure and `mlcroissant`, you can programmatically access, filter, and analyze rich, metadata-driven datasets reproducibly. For more advanced processing, simply follow the Croissant schema's `@id` references and field documentation for robust downstream analyses.